# Properties: dipole to IR

The response-property layer: dipole, polarizability, harmonic frequencies and IR intensities, and alchemical derivatives, all from the same autodiff engine.

In [1]:
import jax; jax.config.update("jax_enable_x64", True)
import numpy as np
from dftax import Molecule, becke, dipole, polarizability, ir_spectrum, alchemical_deriv
from dftax.energy.xc import PBE

mol = Molecule.from_xyz("O 0 0 0; H 0.7586 0 0.5043; H 0.7586 0 -0.5043", "sto-3g")

mu = np.asarray(dipole(mol, PBE(), debye=True))
print(f"dipole moment: {np.linalg.norm(mu):.4f} Debye  {np.round(mu, 4)}")

alpha = np.asarray(polarizability(mol, PBE()))                    # finite field
print(f"isotropic polarizability (FD):       {np.trace(alpha) / 3:.4f} a.u.")
alpha_a = np.asarray(polarizability(mol, PBE(), method="analytic"))  # implicit-diff CPHF
print(f"isotropic polarizability (analytic): {np.trace(alpha_a) / 3:.4f} a.u.")

ir = ir_spectrum(mol, PBE(), grid=becke(60, 194))
freq = np.asarray(ir.frequencies); inten = np.asarray(ir.intensities)
print("\nIR-active modes (vibrational):")
for f, a in sorted(zip(freq, inten)):
    if f > 100:  # skip the ~0 translations/rotations
        print(f"  {f:8.1f} cm^-1   {a:7.2f} km/mol")

dEdZ = np.asarray(alchemical_deriv(mol, PBE()))
print("\nalchemical gradient dE/dZ (Ha/charge):", np.round(dEdZ, 4))

dipole moment: 1.9240 Debye  [ 1.924 -0.     0.   ]


isotropic polarizability (FD):       2.1318 a.u.


isotropic polarizability (analytic): 2.1318 a.u.



IR-active modes (vibrational):
    2377.3 cm^-1      0.00 km/mol
    5026.9 cm^-1      0.04 km/mol
    5456.3 cm^-1      0.06 km/mol



alchemical gradient dE/dZ (Ha/charge): [-22.0529  -1.0029  -1.0029]
